**Data Model & Table Creation**

In [0]:
-- Create a database for DFPS data
CREATE DATABASE IF NOT EXISTS dfps_validation
COMMENT 'Database for DFPS case data and validation';


In [0]:
USE dfps_validation;

**Create Source Data Table** <br>
Create Vendors Table

In [0]:
-- Vendors/Service Providers Table
CREATE TABLE IF NOT EXISTS vendors (
    vendor_id STRING,
    vendor_name STRING,
    vendor_type STRING,
    ein_tax_id STRING,
    business_address STRING,
    city STRING,
    state STRING,
    zip_code STRING,
    contact_name STRING,
    contact_email STRING,
    contact_phone STRING,
    vendor_status STRING,
    registration_date DATE,
    last_audit_date DATE,
    is_minority_owned BOOLEAN,
    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'DFPS approved vendors and service providers';

Create Contracts Table

In [0]:
-- Contracts Table
CREATE TABLE IF NOT EXISTS contracts (
    contract_id STRING,
    contract_number STRING,
    vendor_id STRING,
    contract_type STRING,
    service_category STRING,
    contract_description STRING,
    contract_amount DECIMAL(15,2),
    start_date DATE,
    end_date DATE,
    contract_status STRING,
    region STRING,
    program_code STRING,
    fund_source STRING,
    requires_performance_bond BOOLEAN,
    bond_amount DECIMAL(15,2),
    executed_date DATE,
    procurement_method STRING,
    contract_manager_id STRING,
    contract_manager_name STRING,
    created_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'DFPS contracts with service providers';

Create Expenditures Table

In [0]:
-- Expenditures/Payments Table
CREATE TABLE IF NOT EXISTS expenditures (
    expenditure_id STRING,
    contract_id STRING,
    vendor_id STRING,
    invoice_number STRING,
    invoice_date DATE,
    payment_date DATE,
    expenditure_amount DECIMAL(15,2),
    fiscal_year INT,
    fiscal_quarter STRING,
    expense_category STRING,
    service_period_start DATE,
    service_period_end DATE,
    case_count INT,
    payment_status STRING,
    approved_by STRING,
    approval_date DATE,
    payment_method STRING,
    gl_account STRING,
    notes STRING,
    created_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'DFPS expenditures and vendor payments';

Create Invoices Table

In [0]:
-- Invoices Table (detailed line items)
CREATE TABLE IF NOT EXISTS invoices (
    invoice_id STRING,
    invoice_number STRING,
    contract_id STRING,
    vendor_id STRING,
    invoice_date DATE,
    due_date DATE,
    invoice_amount DECIMAL(15,2),
    invoice_status STRING,
    service_description STRING,
    number_of_children INT,
    daily_rate DECIMAL(10,2),
    number_of_days INT,
    submitted_date DATE,
    reviewed_by STRING,
    review_date DATE,
    has_supporting_docs BOOLEAN,
    payment_terms STRING,
    created_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'DFPS vendor invoices awaiting processing';

Create Validation Rules Table

In [0]:
-- Validation Rules Configuration
CREATE TABLE IF NOT EXISTS validation_rules (
    rule_id INT,
    rule_name STRING,
    rule_category STRING,
    rule_description STRING,
    validation_query STRING,
    severity STRING,
    regulatory_reference STRING,
    is_active BOOLEAN
)
USING DELTA;

Create Validation Results Table

In [0]:
-- Validation Results Log
CREATE TABLE IF NOT EXISTS validation_results (
    validation_id STRING,
    rule_id INT,
    rule_name STRING,
    rule_category STRING,
    execution_timestamp TIMESTAMP,
    records_checked INT,
    records_failed INT,
    failure_rate DECIMAL(5,2),
    status STRING,
    error_details STRING,
    validation_date DATE
)
USING DELTA
PARTITIONED BY (validation_date);

Create Failed Records Table

In [0]:
-- Failed Validation Records
CREATE TABLE IF NOT EXISTS failed_validation_records (
    validation_id STRING,
    rule_id INT,
    record_type STRING,
    record_id STRING,
    record_identifier STRING,
    failure_reason STRING,
    financial_impact DECIMAL(15,2),
    risk_level STRING,
    failure_timestamp TIMESTAMP
)
USING DELTA
PARTITIONED BY (failure_timestamp);

**Data Ingestion** <br>

Insert Vendor Data

In [0]:
-- Insert sample vendor data (with data quality issues)
INSERT INTO vendors VALUES
('V001', 'ABC Foster Care Services', 'Foster Care Agency', '74-1234567', '123 Main St', 'Austin', 'TX', '78701', 'John Smith', 'john@abcfoster.com', '512-555-0101', 'Active', '2020-01-15', '2024-06-30', true, current_timestamp(), current_timestamp()),
('V002', 'Hope Family Services', 'Residential Treatment', '75-2345678', '456 Oak Ave', 'Houston', 'TX', '77001', 'Sarah Johnson', 'sarah@hopefamily.org', '713-555-0202', 'Active', '2019-03-20', '2024-08-15', false, current_timestamp(), current_timestamp()),
('V003', 'Bright Futures Inc', 'Foster Care Agency', NULL, '789 Elm St', 'Dallas', 'TX', '75201', 'Mike Williams', NULL, '214-555-0303', 'Active', '2021-07-10', '2023-12-01', false, current_timestamp(), current_timestamp()),
('V004', 'Safe Haven Services', 'Emergency Shelter', '76-3456789', '321 Pine Rd', 'San Antonio', 'TX', '78201', NULL, 'contact@safehaven.org', '210-555-0404', 'Active', '2018-11-05', '2024-09-20', true, current_timestamp(), current_timestamp()),
('V005', 'Kids First Agency', 'Foster Care Agency', '77-4567890', '654 Cedar Ln', 'Fort Worth', 'TX', '76101', 'Lisa Brown', 'lisa@kidsfirst.com', NULL, 'Suspended', '2022-02-28', '2024-01-10', false, current_timestamp(), current_timestamp()),
('V006', 'Family Solutions LLC', 'Counseling Services', '78-5678901', '987 Maple Dr', 'Austin', 'TX', '78702', 'Tom Davis', 'tom@familysolutions.org', '512-555-0606', 'Active', '2020-09-15', NULL, false, current_timestamp(), current_timestamp()),
('V007', 'New Horizons Foster Care', 'Foster Care Agency', '79-6789012', '147 Birch St', 'Houston', 'TX', '77002', 'Emily Clark', 'emily@newhorizons.com', '713-555-0707', 'Active', '2021-04-12', '2024-07-22', true, current_timestamp(), current_timestamp()),
('V008', NULL, 'Residential Treatment', '80-7890123', '258 Spruce Ave', 'Dallas', 'TX', '75202', 'David Lee', 'david@unknown.org', '214-555-0808', 'Active', '2023-01-05', '2024-10-30', false, current_timestamp(), current_timestamp());

Insert Contract Data

In [0]:
-- Insert sample contract data (with validation issues)
INSERT INTO contracts VALUES
('C001', 'DFPS-FY25-001', 'V001', 'Foster Care Services', 'Basic Foster Care', 'Daily care for foster children', 250000.00, '2024-09-01', '2025-08-31', 'Active', 'Central', 'FC-001', 'State General Revenue', true, 25000.00, '2024-08-15', 'Competitive Bid', 'MGR001', 'Amanda White', current_timestamp(), current_timestamp()),
('C002', 'DFPS-FY25-002', 'V002', 'Residential Treatment', 'Level 4 Residential', 'High-needs residential treatment', 500000.00, '2024-09-01', '2025-08-31', 'Active', 'Southeast', 'RT-002', 'Federal TANF', true, 50000.00, '2024-08-20', 'Sole Source', 'MGR002', 'Robert Green', current_timestamp(), current_timestamp()),
('C003', NULL, 'V003', 'Foster Care Services', 'Therapeutic Foster Care', 'Specialized therapeutic foster care', 180000.00, '2024-10-01', '2025-09-30', 'Active', 'North', 'TFC-003', 'State General Revenue', false, NULL, '2024-09-25', 'Competitive Bid', 'MGR001', 'Amanda White', current_timestamp(), current_timestamp()),
('C004', 'DFPS-FY25-004', 'V004', 'Emergency Services', 'Emergency shelter care', 'Short-term emergency placements', 150000.00, '2024-09-01', '2025-08-31', 'Active', 'Southwest', 'ES-004', 'State General Revenue', true, 15000.00, NULL, 'Emergency Procurement', 'MGR003', 'Jennifer Lopez', current_timestamp(), current_timestamp()),
('C005', 'DFPS-FY25-005', 'V005', 'Foster Care Services', 'Basic Foster Care', 'Standard foster care services', 200000.00, '2024-07-01', '2024-12-31', 'Terminated', 'West', 'FC-001', 'State General Revenue', true, 20000.00, '2024-06-20', 'Competitive Bid', 'MGR002', 'Robert Green', current_timestamp(), current_timestamp()),
('C006', 'DFPS-FY25-006', 'V999', 'Counseling Services', 'Family therapy and counseling', 'Mental health services', 120000.00, '2024-09-01', '2025-08-31', 'Active', 'Central', 'CS-005', 'Federal Medicaid', false, NULL, '2024-08-28', 'Small Purchase', 'MGR004', 'Carlos Martinez', current_timestamp(), current_timestamp()),
('C007', 'DFPS-FY25-007', 'V007', 'Foster Care Services', 'Basic Foster Care', 'Daily care for foster children', 300000.00, '2024-09-01', NULL, 'Active', 'Southeast', 'FC-001', 'State General Revenue', true, 30000.00, '2024-08-18', 'Competitive Bid', 'MGR001', 'Amanda White', current_timestamp(), current_timestamp()),
('C008', 'DFPS-FY24-008', 'V002', 'Residential Treatment', 'Level 3 Residential', 'Medium-needs residential treatment', 400000.00, '2023-09-01', '2024-08-31', 'Expired', 'North', 'RT-003', 'Federal TANF', true, 40000.00, '2023-08-15', 'Competitive Bid', 'MGR002', 'Robert Green', current_timestamp(), current_timestamp());

Insert Expenditure Data

In [0]:
-- Insert sample expenditure data (with validation issues)
INSERT INTO expenditures VALUES
('E001', 'C001', 'V001', 'INV-2025-001', '2025-01-15', '2025-01-30', 18500.00, 2025, 'Q2', 'Foster Care Services', '2025-01-01', '2025-01-31', 25, 'Paid', 'Amanda White', '2025-01-20', 'ACH', '5100-001', 'Regular monthly payment', current_timestamp()),
('E002', 'C002', 'V002', 'INV-2025-002', '2025-01-16', '2025-02-01', 42000.00, 2025, 'Q2', 'Residential Treatment', '2025-01-01', '2025-01-31', 12, 'Paid', 'Robert Green', '2025-01-22', 'ACH', '5200-002', 'Regular monthly payment', current_timestamp()),
('E003', 'C003', 'V003', 'INV-2025-003', '2025-01-20', '2025-02-05', 15200.00, 2025, 'Q2', 'Therapeutic Foster Care', '2025-01-01', '2025-01-31', 20, 'Pending Approval', NULL, NULL, 'ACH', '5100-003', 'Awaiting manager approval', current_timestamp()),
('E004', 'C001', 'V001', 'INV-2025-004', '2025-02-01', '2025-02-15', 95000.00, 2025, 'Q2', 'Foster Care Services', '2025-02-01', '2025-02-28', 28, 'Paid', 'Amanda White', '2025-02-10', 'ACH', '5100-001', 'ALERT: Exceeds monthly expected', current_timestamp()),
('E005', 'C008', 'V002', 'INV-2025-005', '2025-01-25', NULL, 38000.00, 2025, 'Q2', 'Residential Treatment', '2025-01-01', '2025-01-31', 10, 'Pending Payment', 'Robert Green', '2025-01-28', 'ACH', '5200-003', 'Payment against expired contract', current_timestamp()),
('E006', 'C004', 'V004', NULL, '2025-01-18', '2025-02-02', 12500.00, 2025, 'Q2', 'Emergency Services', '2025-01-01', '2025-01-31', 15, 'Paid', 'Jennifer Lopez', '2025-01-25', 'Check', '5300-004', 'Missing invoice number', current_timestamp()),
('E007', 'C002', 'V002', 'INV-2025-007', '2025-02-05', '2025-02-20', 41500.00, 2025, 'Q2', 'Residential Treatment', '2025-02-01', '2025-02-28', 11, 'Paid', 'Robert Green', '2025-02-12', 'ACH', '5200-002', 'Regular monthly payment', current_timestamp()),
('E008', 'C005', 'V005', 'INV-2025-008', '2025-01-10', '2025-01-25', 16800.00, 2025, 'Q2', 'Foster Care Services', '2025-01-01', '2025-01-31', 22, 'Paid', 'Robert Green', '2025-01-18', 'ACH', '5100-005', 'Payment on terminated contract', current_timestamp()),
('E009', 'C999', 'V001', 'INV-2025-009', '2025-02-08', NULL, 22000.00, 2025, 'Q2', 'Foster Care Services', '2025-02-01', '2025-02-28', 30, 'Pending Approval', NULL, NULL, 'ACH', '5100-001', 'Contract ID does not exist', current_timestamp()),
('E010', 'C001', 'V002', 'INV-2025-010', '2025-02-10', '2025-02-25', 19500.00, 2025, 'Q2', 'Foster Care Services', '2025-02-01', '2025-02-28', 26, 'Paid', 'Amanda White', '2025-02-18', 'ACH', '5100-001', 'Vendor mismatch with contract', current_timestamp());

Insert Invoice Data

In [0]:
-- Insert sample invoice data
INSERT INTO invoices VALUES
('INV001', 'INV-2025-001', 'C001', 'V001', '2025-01-15', '2025-02-14', 18500.00, 'Paid', 'Foster care services for 25 children', 25, 24.67, 30, '2025-01-15', 'Amanda White', '2025-01-18', true, 'Net 30', current_timestamp()),
('INV002', 'INV-2025-002', 'C002', 'V002', '2025-01-16', '2025-02-15', 42000.00, 'Paid', 'Residential treatment for 12 children', 12, 116.67, 30, '2025-01-16', 'Robert Green', '2025-01-19', true, 'Net 30', current_timestamp()),
('INV003', 'INV-2025-003', 'C003', 'V003', '2025-01-20', '2025-02-19', 15200.00, 'Under Review', 'Therapeutic foster care services', 20, 25.33, 30, '2025-01-20', 'Amanda White', '2025-01-23', false, 'Net 30', current_timestamp()),
('INV004', 'INV-2025-004', 'C001', 'V001', '2025-02-01', '2025-03-03', 95000.00, 'Paid', 'Foster care services - unusual amount', 28, 113.10, 30, '2025-02-01', 'Amanda White', '2025-02-05', true, 'Net 30', current_timestamp()),
('INV005', 'INV-2025-011', 'C001', 'V001', '2025-02-12', '2025-03-14', 21000.00, 'Submitted', 'Foster care services for February', 27, 25.93, 30, '2025-02-12', NULL, NULL, true, 'Net 30', current_timestamp());

Insert Validation Rules

In [0]:
-- Insert comprehensive validation rules
INSERT INTO validation_rules VALUES
(1, 'Missing Vendor Tax ID', 'Vendor Compliance', 'Check for vendors without EIN/Tax ID', 
 'SELECT COUNT(*) FROM vendors WHERE ein_tax_id IS NULL AND vendor_status = "Active"', 
 'Critical', 'IRS Requirements', true),

(2, 'Missing Vendor Contact Email', 'Vendor Compliance', 'Check for active vendors without contact email', 
 'SELECT COUNT(*) FROM vendors WHERE contact_email IS NULL AND vendor_status = "Active"', 
 'High', 'DFPS Vendor Policy 3.2', true),

(3, 'Vendor Audit Overdue', 'Vendor Compliance', 'Check for vendors with audits older than 1 year', 
 'SELECT COUNT(*) FROM vendors WHERE vendor_status = "Active" AND (last_audit_date IS NULL OR DATEDIFF(CURRENT_DATE(), last_audit_date) > 365)', 
 'High', 'DFPS Audit Requirements', true),

(4, 'Missing Contract Number', 'Contract Integrity', 'Check for contracts without contract numbers', 
 'SELECT COUNT(*) FROM contracts WHERE contract_number IS NULL AND contract_status = "Active"', 
 'Critical', 'State Procurement Code', true),

(5, 'Missing Contract Execution Date', 'Contract Integrity', 'Check for active contracts without execution dates', 
 'SELECT COUNT(*) FROM contracts WHERE executed_date IS NULL AND contract_status = "Active"', 
 'High', 'Contract Management Standards', true),

(6, 'Contract Missing End Date', 'Contract Integrity', 'Check for active contracts without end dates', 
 'SELECT COUNT(*) FROM contracts WHERE end_date IS NULL AND contract_status = "Active"', 
 'High', 'Contract Management Standards', true),

(7, 'Orphaned Contracts', 'Referential Integrity', 'Check for contracts with invalid vendor IDs', 
 'SELECT COUNT(*) FROM contracts c WHERE NOT EXISTS (SELECT 1 FROM vendors v WHERE v.vendor_id = c.vendor_id)', 
 'Critical', 'Data Integrity Standards', true),

(8, 'Contracts with Suspended Vendors', 'Vendor Compliance', 'Check for active contracts with suspended vendors', 
 'SELECT COUNT(*) FROM contracts c JOIN vendors v ON c.vendor_id = v.vendor_id WHERE c.contract_status = "Active" AND v.vendor_status = "Suspended"', 
 'Critical', 'DFPS Vendor Policy 5.1', true),

(9, 'Expired Active Contracts', 'Contract Integrity', 'Check for contracts marked Active but past end date', 
 'SELECT COUNT(*) FROM contracts WHERE contract_status = "Active" AND end_date < CURRENT_DATE()', 
 'High', 'Contract Lifecycle Management', true),

(10, 'Missing Performance Bond', 'Contract Compliance', 'Check for contracts requiring bonds but missing bond amounts', 
 'SELECT COUNT(*) FROM contracts WHERE requires_performance_bond = true AND (bond_amount IS NULL OR bond_amount = 0)', 
 'High', 'State Bonding Requirements', true),

(11, 'Expenditure Missing Invoice', 'Payment Integrity', 'Check for expenditures without invoice numbers', 
 'SELECT COUNT(*) FROM expenditures WHERE invoice_number IS NULL', 
 'Critical', 'Invoice Processing Standards', true),

(12, 'Payment Without Approval', 'Payment Integrity', 'Check for paid expenditures without approval', 
 'SELECT COUNT(*) FROM expenditures WHERE payment_status = "Paid" AND (approved_by IS NULL OR approval_date IS NULL)', 
 'Critical', 'Financial Authorization Policy', true),

(13, 'Expenditure Against Expired Contract', 'Contract Compliance', 'Check for payments on expired contracts', 
 'SELECT COUNT(*) FROM expenditures e JOIN contracts c ON e.contract_id = c.contract_id WHERE c.contract_status = "Expired" AND e.payment_date >= c.end_date', 
 'Critical', 'Contract Authority Limits', true),

(14, 'Expenditure Against Terminated Contract', 'Contract Compliance', 'Check for payments on terminated contracts', 
 'SELECT COUNT(*) FROM expenditures e JOIN contracts c ON e.contract_id = c.contract_id WHERE c.contract_status = "Terminated"', 
 'Critical', 'Contract Authority Limits', true),

(15, 'Vendor Mismatch', 'Referential Integrity', 'Check for expenditures where vendor does not match contract vendor', 
 'SELECT COUNT(*) FROM expenditures e JOIN contracts c ON e.contract_id = c.contract_id WHERE e.vendor_id != c.vendor_id', 
 'Critical', 'Data Integrity Standards', true),

(16, 'Contract Overspending', 'Financial Controls', 'Check for contracts where total expenditures exceed contract amount', 
 'SELECT COUNT(*) FROM (SELECT c.contract_id, c.contract_amount, SUM(e.expenditure_amount) as total_spent FROM contracts c LEFT JOIN expenditures e ON c.contract_id = e.contract_id WHERE c.contract_status IN ("Active", "Expired") GROUP BY c.contract_id, c.contract_amount HAVING total_spent > contract_amount)', 
 'Critical', 'Budget Authority Limits', true),

(17, 'Large Payment Anomaly', 'Financial Controls', 'Check for unusually large payments (>50% of monthly average)', 
 'SELECT COUNT(*) FROM expenditures WHERE expenditure_amount > 50000', 
 'Medium', 'Anomaly Detection Standards', true),

(18, 'Missing Supporting Documentation', 'Invoice Compliance', 'Check for invoices without supporting documentation', 
 'SELECT COUNT(*) FROM invoices WHERE has_supporting_docs = false AND invoice_status != "Rejected"', 
 'High', 'Invoice Processing Standards', true),

(19, 'Orphaned Expenditures', 'Referential Integrity', 'Check for expenditures with invalid contract IDs', 
 'SELECT COUNT(*) FROM expenditures e WHERE NOT EXISTS (SELECT 1 FROM contracts c WHERE c.contract_id = e.contract_id)', 
 'Critical', 'Data Integrity Standards', true),

(20, 'Unreviewed Invoices Over 30 Days', 'Processing Efficiency', 'Check for invoices submitted over 30 days ago without review', 
 'SELECT COUNT(*) FROM invoices WHERE invoice_status = "Submitted" AND DATEDIFF(CURRENT_DATE(), submitted_date) > 30', 
 'Medium', 'Invoice Processing SLA', true);

**Validation Execution Logic**

Create Master Validation Procedure

In [0]:
-- ============================================================
-- VALIDATION RUN SETUP (single row)
-- ============================================================

CREATE OR REPLACE TEMP VIEW current_validation AS
SELECT
  uuid()              AS validation_id,
  current_timestamp() AS execution_time,
  current_date()      AS validation_date;

-- ============================================================
-- RULE 1: Missing Vendor Tax ID
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  1 AS rule_id,
  'Missing Vendor Tax ID' AS rule_name,
  'Vendor Compliance'     AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active') AS records_checked,
  (SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active' AND v.ein_tax_id IS NULL) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active' AND v.ein_tax_id IS NULL) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active'), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active' AND v.ein_tax_id IS NULL) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  CONCAT(
    'Found ',
    (SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active' AND v.ein_tax_id IS NULL),
    ' active vendors without Tax ID'
  ) AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  1 AS rule_id,
  'Vendor' AS record_type,
  v.vendor_id   AS record_id,
  v.vendor_name AS record_identifier,
  'Active vendor is missing EIN/Tax ID' AS failure_reason,
  CAST(NULL AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM vendors v
CROSS JOIN current_validation cv
WHERE v.vendor_status = 'Active' AND v.ein_tax_id IS NULL;


-- ============================================================
-- RULE 3: Vendor Audit Overdue
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  3 AS rule_id,
  'Vendor Audit Overdue' AS rule_name,
  'Vendor Compliance'    AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active') AS records_checked,
  (SELECT COUNT(*) FROM vendors v
    WHERE v.vendor_status = 'Active'
      AND (v.last_audit_date IS NULL OR DATEDIFF(CURRENT_DATE(), v.last_audit_date) > 365)
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM vendors v
      WHERE v.vendor_status = 'Active'
        AND (v.last_audit_date IS NULL OR DATEDIFF(CURRENT_DATE(), v.last_audit_date) > 365)
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM vendors v WHERE v.vendor_status = 'Active'), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM vendors v
      WHERE v.vendor_status = 'Active'
        AND (v.last_audit_date IS NULL OR DATEDIFF(CURRENT_DATE(), v.last_audit_date) > 365)
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Active vendors must be audited at least annually' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  3 AS rule_id,
  'Vendor' AS record_type,
  v.vendor_id   AS record_id,
  v.vendor_name AS record_identifier,
  CONCAT(
    'Last audit: ',
    COALESCE(CAST(v.last_audit_date AS STRING), 'Never'),
    ' - Overdue for annual audit'
  ) AS failure_reason,
  CAST(NULL AS DECIMAL(15,2)) AS financial_impact,
  'High' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM vendors v
CROSS JOIN current_validation cv
WHERE v.vendor_status = 'Active'
  AND (v.last_audit_date IS NULL OR DATEDIFF(CURRENT_DATE(), v.last_audit_date) > 365);


-- ============================================================
-- RULE 4: Missing Contract Number
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  4 AS rule_id,
  'Missing Contract Number' AS rule_name,
  'Contract Integrity'       AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active') AS records_checked,
  (SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active' AND c.contract_number IS NULL) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active' AND c.contract_number IS NULL) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active'), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active' AND c.contract_number IS NULL) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'All active contracts must have contract numbers' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  4 AS rule_id,
  'Contract' AS record_type,
  c.contract_id AS record_id,
  CONCAT('Contract with ', CAST(c.vendor_id AS STRING), ' for ', COALESCE(c.service_category,'(unknown)')) AS record_identifier,
  'Active contract missing contract number' AS failure_reason,
  CAST(c.contract_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM contracts c
CROSS JOIN current_validation cv
WHERE c.contract_status = 'Active' AND c.contract_number IS NULL;


-- ============================================================
-- RULE 7: Orphaned Contracts  (REWRITE: LEFT ANTI JOIN)
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  7 AS rule_id,
  'Orphaned Contracts'    AS rule_name,
  'Referential Integrity' AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM contracts c) AS records_checked,
  (SELECT COUNT(*) FROM contracts c
    LEFT ANTI JOIN vendors v
      ON v.vendor_id = c.vendor_id
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM contracts c
      LEFT ANTI JOIN vendors v
        ON v.vendor_id = c.vendor_id
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM contracts c), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM contracts c
      LEFT ANTI JOIN vendors v
        ON v.vendor_id = c.vendor_id
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Contracts reference non-existent vendors' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  7 AS rule_id,
  'Contract' AS record_type,
  c.contract_id     AS record_id,
  c.contract_number AS record_identifier,
  CONCAT('References invalid vendor_id: ', CAST(c.vendor_id AS STRING)) AS failure_reason,
  CAST(c.contract_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM (
  SELECT c.*
  FROM contracts c
  LEFT ANTI JOIN vendors v
    ON v.vendor_id = c.vendor_id
) c
CROSS JOIN current_validation cv;


-- ============================================================
-- RULE 8: Contracts with Suspended Vendors
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  8 AS rule_id,
  'Contracts with Suspended Vendors' AS rule_name,
  'Vendor Compliance'                AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active') AS records_checked,
  (SELECT COUNT(*) FROM contracts c
     JOIN vendors v ON v.vendor_id = c.vendor_id
    WHERE c.contract_status = 'Active' AND v.vendor_status = 'Suspended'
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM contracts c
       JOIN vendors v ON v.vendor_id = c.vendor_id
      WHERE c.contract_status = 'Active' AND v.vendor_status = 'Suspended'
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM contracts c WHERE c.contract_status = 'Active'), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM contracts c
       JOIN vendors v ON v.vendor_id = c.vendor_id
      WHERE c.contract_status = 'Active' AND v.vendor_status = 'Suspended'
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Active contracts must not have suspended vendors' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  8 AS rule_id,
  'Contract' AS record_type,
  c.contract_id     AS record_id,
  c.contract_number AS record_identifier,
  CONCAT('Vendor ', v.vendor_name, ' is suspended but contract is active') AS failure_reason,
  CAST(c.contract_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM contracts c
JOIN vendors v ON v.vendor_id = c.vendor_id
CROSS JOIN current_validation cv
WHERE c.contract_status = 'Active' AND v.vendor_status = 'Suspended';


-- ============================================================
-- RULE 11: Expenditure Missing Invoice
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  11 AS rule_id,
  'Expenditure Missing Invoice' AS rule_name,
  'Payment Integrity'           AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM expenditures e) AS records_checked,
  (SELECT COUNT(*) FROM expenditures e WHERE e.invoice_number IS NULL) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM expenditures e WHERE e.invoice_number IS NULL) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM expenditures e), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM expenditures e WHERE e.invoice_number IS NULL) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'All expenditures require invoice numbers' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  11 AS rule_id,
  'Expenditure' AS record_type,
  e.expenditure_id AS record_id,
  CONCAT('Payment to ', CAST(e.vendor_id AS STRING), ' - $', CAST(e.expenditure_amount AS STRING)) AS record_identifier,
  'Expenditure processed without invoice number' AS failure_reason,
  CAST(e.expenditure_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM expenditures e
CROSS JOIN current_validation cv
WHERE e.invoice_number IS NULL;


-- ============================================================
-- RULE 13: Expenditure Against Expired Contract
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  13 AS rule_id,
  'Expenditure Against Expired Contract' AS rule_name,
  'Contract Compliance'                  AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM expenditures e) AS records_checked,
  (SELECT COUNT(*) FROM expenditures e
     JOIN contracts c ON c.contract_id = e.contract_id
    WHERE c.contract_status = 'Expired'
      AND e.payment_date >= c.end_date
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM expenditures e
       JOIN contracts c ON c.contract_id = e.contract_id
      WHERE c.contract_status = 'Expired'
        AND e.payment_date >= c.end_date
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM expenditures e), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM expenditures e
       JOIN contracts c ON c.contract_id = e.contract_id
      WHERE c.contract_status = 'Expired'
        AND e.payment_date >= c.end_date
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Payments made after contract expiration' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  13 AS rule_id,
  'Expenditure' AS record_type,
  e.expenditure_id AS record_id,
  e.invoice_number AS record_identifier,
  CONCAT(
    'Payment of $', CAST(e.expenditure_amount AS STRING),
    ' made on ', CAST(e.payment_date AS STRING),
    ' after contract expired on ', CAST(c.end_date AS STRING)
  ) AS failure_reason,
  CAST(e.expenditure_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM expenditures e
JOIN contracts c ON c.contract_id = e.contract_id
CROSS JOIN current_validation cv
WHERE c.contract_status = 'Expired'
  AND e.payment_date >= c.end_date;


-- ============================================================
-- RULE 14: Expenditure Against Terminated Contract
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  14 AS rule_id,
  'Expenditure Against Terminated Contract' AS rule_name,
  'Contract Compliance'                     AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM expenditures e) AS records_checked,
  (SELECT COUNT(*) FROM expenditures e
     JOIN contracts c ON c.contract_id = e.contract_id
    WHERE c.contract_status = 'Terminated'
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM expenditures e
       JOIN contracts c ON c.contract_id = e.contract_id
      WHERE c.contract_status = 'Terminated'
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM expenditures e), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM expenditures e
       JOIN contracts c ON c.contract_id = e.contract_id
      WHERE c.contract_status = 'Terminated'
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Payments made on terminated contracts' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  14 AS rule_id,
  'Expenditure' AS record_type,
  e.expenditure_id AS record_id,
  e.invoice_number AS record_identifier,
  CONCAT(
    'Payment of $', CAST(e.expenditure_amount AS STRING),
    ' made on terminated contract ', COALESCE(c.contract_number,'(missing)')
  ) AS failure_reason,
  CAST(e.expenditure_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM expenditures e
JOIN contracts c ON c.contract_id = e.contract_id
CROSS JOIN current_validation cv
WHERE c.contract_status = 'Terminated';


-- ============================================================
-- RULE 15: Vendor Mismatch
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  15 AS rule_id,
  'Vendor Mismatch'       AS rule_name,
  'Referential Integrity' AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM expenditures e) AS records_checked,
  (SELECT COUNT(*) FROM expenditures e
     JOIN contracts c ON c.contract_id = e.contract_id
    WHERE e.vendor_id <> c.vendor_id
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM expenditures e
       JOIN contracts c ON c.contract_id = e.contract_id
      WHERE e.vendor_id <> c.vendor_id
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM expenditures e), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM expenditures e
       JOIN contracts c ON c.contract_id = e.contract_id
      WHERE e.vendor_id <> c.vendor_id
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Expenditure vendor does not match contract vendor' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  15 AS rule_id,
  'Expenditure' AS record_type,
  e.expenditure_id AS record_id,
  e.invoice_number AS record_identifier,
  CONCAT(
    'Expenditure vendor ', CAST(e.vendor_id AS STRING),
    ' does not match contract vendor ', CAST(c.vendor_id AS STRING)
  ) AS failure_reason,
  CAST(e.expenditure_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM expenditures e
JOIN contracts c ON c.contract_id = e.contract_id
CROSS JOIN current_validation cv
WHERE e.vendor_id <> c.vendor_id;


-- ============================================================
-- RULE 16: Contract Overspending
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  16 AS rule_id,
  'Contract Overspending' AS rule_name,
  'Financial Controls'    AS rule_category,
  cv.execution_time,
  (SELECT COUNT(DISTINCT c.contract_id) FROM contracts c WHERE c.contract_status IN ('Active','Expired')) AS records_checked,
  (SELECT COUNT(*) FROM (
      SELECT c.contract_id
      FROM contracts c
      LEFT JOIN expenditures e ON e.contract_id = c.contract_id
      WHERE c.contract_status IN ('Active','Expired')
      GROUP BY c.contract_id, c.contract_amount
      HAVING COALESCE(SUM(e.expenditure_amount), 0) > c.contract_amount
  ) t) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM (
        SELECT c.contract_id
        FROM contracts c
        LEFT JOIN expenditures e ON e.contract_id = c.contract_id
        WHERE c.contract_status IN ('Active','Expired')
        GROUP BY c.contract_id, c.contract_amount
        HAVING COALESCE(SUM(e.expenditure_amount), 0) > c.contract_amount
    ) t2) * 100.0 /
    NULLIF((SELECT COUNT(DISTINCT c.contract_id) FROM contracts c WHERE c.contract_status IN ('Active','Expired')), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM (
        SELECT c.contract_id
        FROM contracts c
        LEFT JOIN expenditures e ON e.contract_id = c.contract_id
        WHERE c.contract_status IN ('Active','Expired')
        GROUP BY c.contract_id, c.contract_amount
        HAVING COALESCE(SUM(e.expenditure_amount), 0) > c.contract_amount
    ) t3) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Contracts exceeded authorized amount' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  16 AS rule_id,
  'Contract' AS record_type,
  c.contract_id     AS record_id,
  c.contract_number AS record_identifier,
  CONCAT(
    'Total spent: $', CAST(SUM(e.expenditure_amount) AS STRING),
    ' exceeds contract amount: $', CAST(c.contract_amount AS STRING),
    ' by $', CAST(SUM(e.expenditure_amount) - c.contract_amount AS STRING)
  ) AS failure_reason,
  CAST(SUM(e.expenditure_amount) - c.contract_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM contracts c
JOIN expenditures e ON e.contract_id = c.contract_id
CROSS JOIN current_validation cv
WHERE c.contract_status IN ('Active','Expired')
GROUP BY cv.validation_id, c.contract_id, c.contract_number, c.contract_amount, cv.execution_time
HAVING SUM(e.expenditure_amount) > c.contract_amount;


-- ============================================================
-- RULE 19: Orphaned Expenditures  (REWRITE: LEFT ANTI JOIN)
-- ============================================================

INSERT INTO validation_results
SELECT
  cv.validation_id,
  19 AS rule_id,
  'Orphaned Expenditures' AS rule_name,
  'Referential Integrity' AS rule_category,
  cv.execution_time,
  (SELECT COUNT(*) FROM expenditures e) AS records_checked,
  (SELECT COUNT(*) FROM expenditures e
     LEFT ANTI JOIN contracts c
       ON c.contract_id = e.contract_id
  ) AS records_failed,
  ROUND(
    (SELECT COUNT(*) FROM expenditures e
       LEFT ANTI JOIN contracts c
         ON c.contract_id = e.contract_id
    ) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM expenditures e), 0),
    2
  ) AS failure_rate,
  CASE
    WHEN (SELECT COUNT(*) FROM expenditures e
       LEFT ANTI JOIN contracts c
         ON c.contract_id = e.contract_id
    ) = 0 THEN 'PASSED'
    ELSE 'FAILED'
  END AS status,
  'Expenditures reference non-existent contracts' AS error_details,
  cv.validation_date
FROM current_validation cv;

INSERT INTO failed_validation_records (
  validation_id, rule_id, record_type, record_id, record_identifier,
  failure_reason, financial_impact, risk_level, failure_timestamp
)
SELECT
  cv.validation_id,
  19 AS rule_id,
  'Expenditure' AS record_type,
  e.expenditure_id AS record_id,
  e.invoice_number  AS record_identifier,
  CONCAT('References invalid contract_id: ', CAST(e.contract_id AS STRING)) AS failure_reason,
  CAST(e.expenditure_amount AS DECIMAL(15,2)) AS financial_impact,
  'Critical' AS risk_level,
  cv.execution_time AS failure_timestamp
FROM (
  SELECT e.*
  FROM expenditures e
  LEFT ANTI JOIN contracts c
    ON c.contract_id = e.contract_id
) e
CROSS JOIN current_validation cv;


-- ============================================================
-- SUMMARY (no severity column used)
-- ============================================================

SELECT
  rule_name,
  rule_category,
  status,
  records_checked,
  records_failed,
  failure_rate,
  error_details
FROM validation_results
WHERE validation_date = CURRENT_DATE()
ORDER BY failure_rate DESC, records_failed DESC;


**Dashboard & Reporting Queries**

Executive Dashboard Query

In [0]:
-- Overall Financial Data Quality Score
SELECT 
    ROUND(100 - AVG(failure_rate), 2) as data_quality_score,
    COUNT(DISTINCT rule_id) as total_rules_checked,
    SUM(CASE WHEN status = 'PASSED' THEN 1 ELSE 0 END) as rules_passed,
    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) as rules_failed,
    SUM(records_failed) as total_records_with_issues,
    MAX(execution_timestamp) as last_validation_time
FROM validation_results
WHERE validation_date = CURRENT_DATE();

Financial Risk Analysis

In [0]:
-- Financial Risk Summary
SELECT 
    fvr.risk_level,
    COUNT(DISTINCT fvr.record_id) as affected_records,
    COALESCE(SUM(fvr.financial_impact), 0) as total_dollars_at_risk,
    COUNT(DISTINCT fvr.rule_id) as number_of_violations
FROM failed_validation_records fvr
WHERE DATE(fvr.failure_timestamp) = CURRENT_DATE()
GROUP BY fvr.risk_level
ORDER BY 
    CASE fvr.risk_level 
        WHEN 'Critical' THEN 1 
        WHEN 'High' THEN 2 
        WHEN 'Medium' THEN 3 
        ELSE 4 
    END;

Contract Compliance Report

In [0]:
-- Contract Issues Requiring Immediate Attention (Databricks/Spark SQL)
SELECT 
    c.contract_number,
    c.vendor_id,
    v.vendor_name,
    c.contract_amount,
    c.contract_status,
    c.end_date,
    concat_ws('; ', collect_set(fvr.failure_reason)) AS compliance_issues,
    COUNT(DISTINCT fvr.rule_id) AS number_of_issues
FROM contracts c
LEFT JOIN vendors v 
  ON c.vendor_id = v.vendor_id
JOIN failed_validation_records fvr 
  ON c.contract_id = fvr.record_id
WHERE fvr.failure_timestamp >= date_trunc('DAY', current_timestamp())
  AND fvr.failure_timestamp <  date_add(date_trunc('DAY', current_timestamp()), 1)
  AND fvr.record_type = 'Contract'
GROUP BY 
    c.contract_number, c.vendor_id, v.vendor_name, 
    c.contract_amount, c.contract_status, c.end_date
ORDER BY number_of_issues DESC, c.contract_amount DESC;


Vendor Compliance Dashboard

In [0]:
-- Vendor Compliance Status
SELECT 
    v.vendor_id,
    v.vendor_name,
    v.vendor_type,
    v.vendor_status,
    v.last_audit_date,
    DATEDIFF(CURRENT_DATE(), v.last_audit_date) as days_since_audit,
    COUNT(DISTINCT c.contract_id) as active_contracts,
    COALESCE(SUM(c.contract_amount), 0) as total_contract_value,
    COUNT(DISTINCT fvr.rule_id) as compliance_issues
FROM vendors v
LEFT JOIN contracts c ON v.vendor_id = c.vendor_id AND c.contract_status = 'Active'
LEFT JOIN failed_validation_records fvr ON v.vendor_id = fvr.record_id 
    AND fvr.record_type = 'Vendor'
    AND DATE(fvr.failure_timestamp) = CURRENT_DATE()
WHERE v.vendor_status = 'Active'
GROUP BY v.vendor_id, v.vendor_name, v.vendor_type, v.vendor_status, v.last_audit_date
ORDER BY compliance_issues DESC, total_contract_value DESC;

Payment Anomaly Detection

In [0]:
-- Unusual Payment Patterns
SELECT 
    e.expenditure_id,
    e.invoice_number,
    e.vendor_id,
    v.vendor_name,
    e.contract_id,
    c.contract_number,
    e.expenditure_amount,
    c.contract_amount,
    ROUND((e.expenditure_amount / c.contract_amount) * 100, 2) as pct_of_contract,
    e.payment_date,
    e.payment_status,
    CASE 
        WHEN e.expenditure_amount > c.contract_amount * 0.5 THEN 'Large Single Payment'
        WHEN e.expenditure_amount > 50000 THEN 'High Dollar Amount'
        ELSE 'Review Required'
    END as anomaly_type
FROM expenditures e
JOIN contracts c ON e.contract_id = c.contract_id
JOIN vendors v ON e.vendor_id = v.vendor_id
WHERE e.expenditure_amount > 50000
    OR e.expenditure_amount > c.contract_amount * 0.5
ORDER BY e.expenditure_amount DESC;

Validation Trend Analysis

In [0]:
-- 7-Day Validation Trends
SELECT 
    validation_date,
    rule_category,
    COUNT(DISTINCT rule_id) as rules_in_category,
    AVG(failure_rate) as avg_failure_rate,
    SUM(records_failed) as total_failures,
    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) as failed_rules
FROM validation_results
WHERE validation_date >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY validation_date, rule_category
ORDER BY validation_date DESC, avg_failure_rate DESC;

Critical Issues Summary (For Management)

In [0]:
-- Critical Issues Requiring Executive Attention
WITH critical_summary AS (
    SELECT 
        vr.rule_name,
        vr.rule_category,
        vr.records_failed,
        COALESCE(SUM(fvr.financial_impact), 0) as total_financial_impact,
        vr.error_details
    FROM validation_results vr
    LEFT JOIN failed_validation_records fvr 
        ON vr.rule_id = fvr.rule_id 
        AND vr.validation_id = fvr.validation_id
    WHERE vr.validation_date = CURRENT_DATE()
        AND vr.status = 'FAILED'
        AND fvr.risk_level = 'Critical'
    GROUP BY vr.rule_name, vr.rule_category, vr.records_failed, vr.error_details
)
SELECT 
    rule_category,
    rule_name,
    records_failed,
    CONCAT('$', FORMAT_NUMBER(total_financial_impact, 2)) as dollars_at_risk,
    error_details
FROM critical_summary
WHERE records_failed > 0
ORDER BY total_financial_impact DESC, records_failed DESC;